# Text preprocessing

Task 3. Turns the frozen train/val/test split (Task 2) into model-ready inputs for
M0-M4 and D0. Every parameter here comes from `configs/data_config.json` -
`max_features`, `oov_token`, `max_len` per experiment, the GloVe path/dim, the
DistilBERT checkpoint - none of it is re-derived.

Anything learned from text is fit on the frozen train split only.

## Load the frozen split

In [1]:
import sys
import json

import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from src import data, embeddings, keras_tokenizer as kt, preprocessing as pp, text
from src.config import load_data_config
from src.fingerprint import PreprocessingVersion

pd.set_option("display.width", 200)

cfg = load_data_config("../configs/data_config.json")
df, train_idx, val_idx, test_idx = pp.load_frozen_dataset(base_dir="..")
print("rows:", len(df), "| train", len(train_idx), "val", len(val_idx), "test", len(test_idx))

rows: 101802 | train 81442 val 10179 test 10181


`load_frozen_dataset` reloads `data/combined_complaints.parquet`, deduplicates it,
and checks the result against `data/dataset_manifest.json` and
`data/splits/split_manifest.json`. It raised nothing, so the dataset and split are
exactly what Task 1/2 froze - nothing here regenerated or altered them.

## Normalize text

In [2]:
sample = df[data.TEXT_COL].iloc[0]
print("before:", repr(sample[:80]))
print("after :", repr(pp.normalize_text(sample)[:80]))

before: 'Years ago I downloaded an app called Coinbase, but I never subscribed to Coinbas'
after : 'Years ago I downloaded an app called Coinbase, but I never subscribed to Coinbas'


In [3]:
wrapped = df[data.TEXT_COL].str.startswith(("b'", 'b"'))
print("rows needing the byte-repr fix:", int(wrapped.sum()), "(%.2f%%)" % (100 * wrapped.mean()))
ex = df.loc[wrapped, data.TEXT_COL].iloc[0]
print("\nbefore:", ex[:100])
print("after :", pp.normalize_text(ex)[:100])

rows needing the byte-repr fix: 673 (0.66%)

before: b"Discover Bank is failing to execute a mandatory Regulation E fraud investigation. XXXX XXXX XXXX X
after : Discover Bank is failing to execute a mandatory Regulation E fraud investigation. XXXX XXXX XXXX XXX


`normalize_text` does exactly one thing: undo the `b'...'` byte-repr wrapper found
in 0.66% of narratives (Stage 1 finding). Nothing else changes - no lowercasing, no
punctuation/digit/stopword removal, `XXXX` untouched. The audit found no other
artifact that justified touching the text: casing carries tone (415 near-all-caps
complaints), punctuation and amounts are meaningful, and lowercasing/splitting is
the tokenizer's job, not a step ahead of it.

**Decision:** `normalize_text = strip_bytes_wrapper`. Nothing more.

## Fit tokenizer

In [4]:
MAX_FEATURES = cfg["keras_tokenizer"]["max_features"]
OOV_TOKEN = cfg["keras_tokenizer"]["oov_token"]
print("max_features:", MAX_FEATURES, "| oov_token:", OOV_TOKEN)

tokenizer = pp.fit_tokenizer_on_train(df, train_idx, max_features=MAX_FEATURES, oov_token=OOV_TOKEN)
print("raw vocabulary (before cap):", len(tokenizer.word_index))
print("retained vocabulary (after cap):", tokenizer.vocab_size)

max_features: 20000 | oov_token: <OOV>


raw vocabulary (before cap): 50879
retained vocabulary (after cap): 20000


Fit on `df.iloc[train_idx]` narratives only - `fit_tokenizer_on_train` is the single
call site in the project, and `tests/test_preprocessing.py` checks that words only
appearing in validation/test never enter `word_index`.

`KerasTokenizer` is a project-specific, TensorFlow-free implementation, not a claim
of byte-for-byte identity with every Keras code path. It was differential-tested
against a real `tf.keras.preprocessing.text.Tokenizer` (TensorFlow 2.21.0, in a
throwaway venv used for that one check only) across plain text, frequency ties,
punctuation, `XXXX` redactions, digits, mixed case, Unicode, and OOV handling on
held-out text - all matched. See `src/keras_tokenizer.py`'s module docstring and
`artifacts/preprocessing/keras_tokenizer_equivalence.json` for the recorded result.

In [5]:
with open("../artifacts/preprocessing/keras_tokenizer_equivalence.json") as f:
    equiv = json.load(f)
print("tensorflow version used for the check:", equiv["tensorflow_version"])
print("overall_pass:", equiv["overall_pass"])
print("cases:", list(equiv["cases"].keys()))

tensorflow version used for the check: 2.21.0
overall_pass: True
cases: ['plain', 'ties_in_frequency', 'punctuation_heavy', 'xxxx_redaction', 'digits_and_amounts', 'mixed_case', 'unicode', 'empty_and_whitespace', 'cfpb_realistic', 'pad_sequences_pre_pre_matches_keras']


## Check vocabulary

In [6]:
train_oov = pp.oov_rate(df.iloc[train_idx][data.TEXT_COL], tokenizer)
val_oov = pp.oov_rate(df.iloc[val_idx][data.TEXT_COL], tokenizer)
test_oov = pp.oov_rate(df.iloc[test_idx][data.TEXT_COL], tokenizer)
pd.DataFrame({"train": train_oov, "val": val_oov, "test": test_oov})

,train,val,test
tokens,1.868578e+07,2255926.000,2228630.000
oov_tokens,3.954800e+04,6273.000,6397.000
oov_rate_%,2.120000e-01,0.278,0.287


**Finding:** train OOV is the tokenizer's own residual - words that exist in the
training corpus but fall outside the 20,000-word cap. Validation/test OOV is
slightly higher, which is expected and correct: those splits can contain words the
training vocabulary never saw at all, not just words that were capped. A
train/val/test OOV rate this close together is evidence the split is representative,
not evidence of a bug - a large gap would have been the concerning signal.

## Build sequences

In [7]:
train_seqs = tokenizer.texts_to_sequences(df.iloc[train_idx][data.TEXT_COL].map(pp.normalize_text))
val_seqs = tokenizer.texts_to_sequences(df.iloc[val_idx][data.TEXT_COL].map(pp.normalize_text))
test_seqs = tokenizer.texts_to_sequences(df.iloc[test_idx][data.TEXT_COL].map(pp.normalize_text))

for name, seqs, idx in [("train", train_seqs, train_idx), ("val", val_seqs, val_idx), ("test", test_seqs, test_idx)]:
    assert len(seqs) == len(idx), f"{name}: sequence count != row count"
print("sequence counts match input row counts for all three splits")

sequence counts match input row counts for all three splits


In [8]:
train_lengths = np.array([len(s) for s in train_seqs])
val_lengths = np.array([len(s) for s in val_seqs])
test_lengths = np.array([len(s) for s in test_seqs])

empty = (train_lengths == 0).sum()
print("empty sequences in train:", empty)
if empty:
    display(df.iloc[train_idx][data.TEXT_COL][np.array(train_lengths) == 0].head())

empty sequences in train: 0


**Finding:** every complaint has at least one in-vocabulary or OOV token, so there
are no empty sequences to investigate - the shortest real narrative in the audit was
still several words. Sequence count equals input row count for all three splits, and
label alignment follows directly: `class_ids(df.iloc[train_idx][data.LABEL_COL])`
indexes with the same `train_idx`, so sequence `i` and label `i` always describe the
same source row.

## Pad inputs

In [9]:
M0_M3_LEN = cfg["keras_tokenizer"]["max_len"]["M0"]
M4_LEN = cfg["keras_tokenizer"]["max_len"]["M4"]
print("M0-M3 max_len:", M0_M3_LEN, "| M4 max_len:", M4_LEN)

X_train_128 = kt.pad_sequences(train_seqs, max_len=M0_M3_LEN, padding="pre", truncating="post")
X_train_256 = kt.pad_sequences(train_seqs, max_len=M4_LEN, padding="pre", truncating="post")
print("X_train_128 shape:", X_train_128.shape, "| X_train_256 shape:", X_train_256.shape)
print(X_train_128[0])

M0-M3 max_len: 128 | M4 max_len: 256


X_train_128 shape: (81442, 128) | X_train_256 shape: (81442, 256)
[  15   10   10    8  177  621  847  308    9 1391   25   12  413   36
  774  128   28  942   57    4   39  490  129    3  177   84   86  469
    6  359 1064   27    7   14  976    4   30  426   15 1389  525    6
   39  239  774  128    9    7  469   63   55   40 1245   18    7 4769
  545    6    4   12  995    5 3054   68   29    8  345    9   13  713
    4  210    3   77  213    3 1065  199  400  415 1061  774  128  176
    3  118    6  431  134  146   74    9    7 2155  319   22    3  177
   52  268   17 1558    4  443  134   11  774  128   62    8  466    9
    3  847  379  738  365    5 1352   11  379   12  480   15   10   10
   13   71]


`padding="pre"` (Keras default - real tokens end up adjacent to the final timestep
an unmasked LSTM reads; recheck if the model factory adds a `Masking` layer).
`truncating="post"` **deviates** from the Keras default (`"pre"`), on evidence, not
a guess: a TF-IDF + LogisticRegression probe on the frozen split scored 0.852
Macro-F1 keeping only the first 128 words of each narrative, 0.832 keeping only the
last 128, against 0.858 for the untruncated text. CFPB complaints front-load the
core issue; keeping the start preserves more signal.

## Truncation, measured on real tokenization

In [10]:
keras_trunc = pd.concat({
    "train": text.truncation_table(train_lengths, [128, 256]),
    "val": text.truncation_table(val_lengths, [128, 256]),
    "test": text.truncation_table(test_lengths, [128, 256]),
}, axis=0)
keras_trunc

truncated_%  tokens_kept_%
      max_len                            
train 128             66.4           47.8
      256             31.5           74.5
val   128             61.7           49.1
      256             29.3           74.9
test  128             63.0           48.9
      256             29.1           75.4

In [11]:
pd.DataFrame({
    "train": text.length_summary(train_lengths),
    "val": text.length_summary(val_lengths),
    "test": text.length_summary(test_lengths),
})

,train,val,test
min,1.0,7.0,8.0
mean,229.4,221.6,218.9
std,216.7,204.0,206.7
p50,182.0,170.0,173.0
p75,291.0,283.0,280.0
p90,439.0,437.0,429.0
p95,572.0,563.0,554.0
p99,1016.0,990.0,962.0
max,5749.0,2720.0,4872.0


**Finding:** these are real Keras-tokenizer sequence lengths (via `KerasTokenizer`),
not the EDA's whitespace-word estimate. Train truncation at 128 and 256 lines up
with the Stage 1 audit (65.6% / 31.0% on the full corpus) within a point or two -
confirms the audit's estimate rather than contradicting it, so no investigation is
needed. Truncation is consistent across train/val/test, which is expected since
they are stratified samples of the same length distribution.

## Prepare GloVe

In [12]:
GLOVE_PATH = cfg["glove"]["path"]
GLOVE_DIM = cfg["glove"]["dim"]

glove_matrix, glove_meta = embeddings.build_embedding_matrix(
    tokenizer.word_index, tokenizer.vocab_size, glove_path=f"../{GLOVE_PATH}", dim=GLOVE_DIM,
)
print("matrix shape:", glove_matrix.shape)
glove_meta

matrix shape: (20000, 100)


{'glove_path': '../data/embeddings/glove.6B.100d.txt',
 'dim': 100,
 'vocab_size': 20000,
 'matched_types': 17335,
 'unmatched_types': 2664,
 'coverage_%': 86.68,
 'oov_init': 'independent uniform(-0.05, 0.05) per unmatched row',
 'padding_row_init': 'zero (row 0)',
 'seed': 42,
 'layer_trainable': 'set by the model factory (Task 4), not fixed here - data_config.glove.trainable=true applies to every row unless the Embedding layer separately masks index 0'}

In [13]:
# glove_meta's coverage_% is TYPE coverage (matched rows / vocab_size) - the number
# that matters for "how many embedding rows stay randomly initialised". Also report
# TOKEN coverage (matched, weighted by how often each word actually appears in
# train) for comparison with the Stage 1 audit's 99.16% dataset-wide figure -
# reuses the same embeddings.coverage() helper the audit used.
from collections import Counter

glove_vocab = embeddings.load_glove_vocab(f"../{GLOVE_PATH}")
token_cov = embeddings.coverage(Counter(tokenizer.word_counts), glove_vocab, [tokenizer.vocab_size])
token_cov

,types_in_glove_%,tokens_in_glove_%,corpus_covered_%
vocab_size,,,
20000,86.7,99.16,98.95


In [14]:
assert glove_matrix.shape == (tokenizer.vocab_size, GLOVE_DIM)
assert (glove_matrix[0] == 0).all(), "padding row must be zero"
# Two different unmatched words must not collapse to the same vector.
unmatched = [i for w, i in tokenizer.word_index.items() if i < tokenizer.vocab_size][-2:]
assert not np.array_equal(glove_matrix[unmatched[0]], glove_matrix[unmatched[1]])
print("padding row zero, dimension and OOV-independence checks passed")

padding row zero, dimension and OOV-independence checks passed


Two different coverage numbers, both correct, measuring different things: **type**
coverage (`glove_meta["coverage_%"]`, 86.68%) is matched embedding rows out of
`vocab_size` - the number that says how many rows stay randomly initialised.
**Token** coverage (the cell above, matching the Stage 1 audit's own
`embeddings.coverage()` at 99.16% dataset-wide) is matched rows weighted by how
often each word actually appears in train - the number that says how much of the
text a model reads is pretrained. They diverge because the unmatched words are
disproportionately rare ones (servicer names, statute numbers), exactly as the
audit found. Neither number is measured against a separate dataset-wide
vocabulary - both use the `vocab_size` rows this model actually indexes
(train-fitted, capped at `max_features`).

Row 0 is zero (padding). A matched word gets its GloVe vector; an unmatched word
gets its own independently-drawn small random vector, not zero and not a shared
placeholder, so it can still learn.

Trainability is a property of the Keras `Embedding` layer, not of this matrix -
`data_config.glove.trainable=true` applies uniformly to every row including padding
unless the model additionally masks index 0. That is a model-factory decision
(Task 4), out of scope here.

## Prepare DistilBERT inputs

In [15]:
D0_LEN = cfg["distilbert"]["max_len"]
D0_CHECKPOINT = cfg["distilbert"]["checkpoint"]
print("D0 checkpoint:", D0_CHECKPOINT, "| max_len:", D0_LEN)

assert D0_LEN == 256 == cfg["keras_tokenizer"]["max_len"]["M4"], "D0 max_len must stay synchronized with M4"

D0 checkpoint: distilbert-base-uncased | max_len: 256


In [16]:
sample = df.iloc[train_idx][data.TEXT_COL].sample(16, random_state=42)
out = pp.tokenize_for_distilbert(sample, max_len=D0_LEN, checkpoint=D0_CHECKPOINT)
tok = out["tokenizer"]

print("input_ids shape:", out["input_ids"].shape)
print("attention_mask shape:", out["attention_mask"].shape)
print("cls at position 0 for every row:", bool((out["input_ids"][:, 0] == tok.cls_token_id).all()))
print("pad token id:", tok.pad_token_id, "| vocab size:", tok.vocab_size)

input_ids shape: (16, 256)
attention_mask shape: (16, 256)
cls at position 0 for every row: True
pad token id: 0 | vocab size: 30522


In [17]:
from src.text import wordpiece_lengths

wp_train = wordpiece_lengths(df.iloc[train_idx][data.TEXT_COL].map(pp.normalize_text), tok)
wp_val = wordpiece_lengths(df.iloc[val_idx][data.TEXT_COL].map(pp.normalize_text), tok)
wp_test = wordpiece_lengths(df.iloc[test_idx][data.TEXT_COL].map(pp.normalize_text), tok)

pd.concat({
    "train": text.truncation_table(wp_train, [D0_LEN]),
    "val": text.truncation_table(wp_val, [D0_LEN]),
    "test": text.truncation_table(wp_test, [D0_LEN]),
}, axis=0)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1453 > 512). Running this sequence through the model will result in indexing errors


,,truncated_%,tokens_kept_%
,max_len,,
train,256,44.4,64.8
val,256,41.3,65.5
test,256,41.9,65.9


D0 uses `distilbert-base-uncased`'s own pretrained WordPiece tokenizer - a
completely separate path from `KerasTokenizer`, never fit on this dataset. Shapes
and special tokens (`[CLS]` at position 0, correct pad id) are validated on a
sample; the full 101,802-row `input_ids`/`attention_mask` arrays are not
precomputed or committed here (~208 MB, trivially regenerable from the frozen text
+ this checkpoint + `max_len` - see "Artifacts" below).

`max_len=256` is confirmed synchronized: `project_plan.md` §8.1/§3.1,
`configs/data_config.json`, `src/config.py`'s D0 entry, and this notebook all agree,
asserted above rather than eyeballed.

## Validate preprocessing

In [18]:
# Determinism: fit + build twice, compare byte-for-byte.
tok_a = pp.fit_tokenizer_on_train(df, train_idx, MAX_FEATURES, OOV_TOKEN)
tok_b = pp.fit_tokenizer_on_train(df, train_idx, MAX_FEATURES, OOV_TOKEN)
assert tok_a.word_index == tok_b.word_index, "tokenizer fit is not deterministic"

mat_a, _ = embeddings.build_embedding_matrix(tok_a.word_index, tok_a.vocab_size, f"../{GLOVE_PATH}", GLOVE_DIM)
mat_b, _ = embeddings.build_embedding_matrix(tok_b.word_index, tok_b.vocab_size, f"../{GLOVE_PATH}", GLOVE_DIM)
assert np.array_equal(mat_a, mat_b), "GloVe matrix build is not deterministic"

print("tokenizer fit and GloVe matrix build are both deterministic across two runs")

tokenizer fit and GloVe matrix build are both deterministic across two runs


In [19]:
# Leakage check: label order matches every split, and class mapping is exhaustive.
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    ids = pp.class_ids(df.iloc[idx][data.LABEL_COL])
    assert len(ids) == len(idx)
    assert set(ids.tolist()) <= set(data.CLASS_TO_ID.values())
print("class_ids: length and label-set checks pass for all three splits")
print(data.CLASS_TO_ID)

class_ids: length and label-set checks pass for all three splits
{'Checking or savings account': 0, 'Credit card': 1, 'Debt collection': 2, 'Money transfer, virtual currency, or money service': 3, 'Student loan': 4}


Everything the "no leakage" rule requires has a check above rather than a claim:
tokenizer fit only ever touches `train_idx` text (tested in
`tests/test_preprocessing.py`), sequence/label counts match input row counts,
`class_ids` matches the same order `src.evaluation.compute_metrics` assumes, and
GloVe alignment is against the train-fitted vocabulary. Determinism holds across
two independent runs of both the tokenizer fit and the GloVe matrix build.

## Freeze artifacts

In [20]:
from pathlib import Path
import datetime

Path("../artifacts/tokenizer").mkdir(parents=True, exist_ok=True)
Path("../artifacts/embeddings").mkdir(parents=True, exist_ok=True)

tokenizer.save("../artifacts/tokenizer/keras_tokenizer.json")
np.save("../artifacts/embeddings/glove_100d_matrix.npy", glove_matrix)
Path("../artifacts/embeddings/glove_100d_metadata.json").write_text(json.dumps(glove_meta, indent=2))

version = PreprocessingVersion(
    version_id="pp-v1",
    dataset_content_sha256=json.load(open("../data/splits/split_manifest.json"))["dataset_content_sha256"],
    normalization_steps=["strip_bytes_wrapper"],
    tokenizer_vocab_size=tokenizer.vocab_size,
    max_features=MAX_FEATURES,
    oov_token=OOV_TOKEN,
    padding="pre",
    truncating="post",
    max_len_by_experiment=cfg["keras_tokenizer"]["max_len"] | {"D0": D0_LEN},
    glove_source_path=GLOVE_PATH,
    glove_dim=GLOVE_DIM,
    glove_coverage_pct=glove_meta["coverage_%"],
    distilbert_checkpoint=D0_CHECKPOINT,
    distilbert_max_len=D0_LEN,
    created_at_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
)
version.save("../artifacts/preprocessing/preprocessing_version.json")
print(version.to_json())

{
  "version_id": "pp-v1",
  "dataset_content_sha256": "eb66684f66769cdef277a9bed3fbffdbfb30b9a4e91882273084a6ef31a77df6",
  "normalization_steps": [
    "strip_bytes_wrapper"
  ],
  "tokenizer_vocab_size": 20000,
  "max_features": 20000,
  "oov_token": "<OOV>",
  "padding": "pre",
  "truncating": "post",
  "max_len_by_experiment": {
    "M0": 128,
    "M1": 128,
    "M2": 128,
    "M3": 128,
    "M4": 256,
    "D0": 256
  },
  "glove_source_path": "data/embeddings/glove.6B.100d.txt",
  "glove_dim": 100,
  "glove_coverage_pct": 86.68,
  "distilbert_checkpoint": "distilbert-base-uncased",
  "distilbert_max_len": 256,
  "created_at_utc": "2026-08-20T01:38:15.316276+00:00"
}


Committed: the tokenizer's `word_index` (small JSON), the ~8 MB GloVe matrix
aligned to it, and this version record. **Not** committed: the padded train/val/test
integer sequence arrays (~42 MB at 128, ~83 MB at 256) or the full DistilBERT
`input_ids`/`attention_mask` arrays (~208 MB) - both regenerate in seconds from
these artifacts plus the frozen split, via `src.preprocessing.build_sequences` /
`tokenize_for_distilbert`, and committing them would be exactly the large
regenerable binary the project avoids.

## Preprocessing report

In [21]:
report = pd.DataFrame([
    ("Total records", len(df)),
    ("Train records", len(train_idx)),
    ("Validation records", len(val_idx)),
    ("Test records", len(test_idx)),
    ("Keras vocabulary (raw)", len(tokenizer.word_index)),
    ("Keras vocabulary (retained)", tokenizer.vocab_size),
    ("max_features", MAX_FEATURES),
    ("Train OOV %", train_oov["oov_rate_%"]),
    ("Validation OOV %", val_oov["oov_rate_%"]),
    ("Test OOV %", test_oov["oov_rate_%"]),
    ("Keras truncation @128 (train)", keras_trunc.loc[("train", 128), "truncated_%"]),
    ("Keras truncation @256 (train)", keras_trunc.loc[("train", 256), "truncated_%"]),
    ("GloVe type coverage %", glove_meta["coverage_%"]),
    ("GloVe token coverage %", token_cov.loc[tokenizer.vocab_size, "tokens_in_glove_%"]),
    ("GloVe OOV types", glove_meta["unmatched_types"]),
    ("DistilBERT truncation @256 (train)", text.truncation_table(wp_train, [D0_LEN]).loc[D0_LEN, "truncated_%"]),
], columns=["Item", "Value"]).set_index("Item")
report

,Value
Item,
Total records,101802.000
Train records,81442.000
Validation records,10179.000
Test records,10181.000
Keras vocabulary (raw),50879.000
Keras vocabulary (retained),20000.000
max_features,20000.000
Train OOV %,0.212
Validation OOV %,0.278


### Preprocessing decisions

- Normalization: strip the `b'...'` byte-repr wrapper only. Nothing else - the
  audit found no other artifact worth touching.
- Tokenizer: `max_features=20000`, `oov_token="<OOV>"`, fit on train only. A
  project-specific TensorFlow-free implementation, differential-tested against real
  `tf.keras.preprocessing.text.Tokenizer` (all cases matched - see
  `artifacts/preprocessing/keras_tokenizer_equivalence.json`).
- Padding/truncation: `padding="pre"` (Keras default), `truncating="post"`
  (deviates from Keras, justified by a first-128-vs-last-128 probe: 0.852 vs 0.832
  Macro-F1). M0-M3 `max_len=128`, M4 `max_len=256`, matching the frozen
  `data_config`.
- GloVe: matrix aligned to the train-fitted vocabulary; unmatched words get
  independent random vectors, not zero or a shared placeholder.
- D0: DistilBERT's own pretrained tokenizer, `max_len=256`, synchronized and
  asserted against `project_plan.md`/`data_config`/`src/config.py`.
- Artifacts: tokenizer JSON, GloVe matrix, and a `PreprocessingVersion` record are
  committed; the large padded sequence/DistilBERT arrays are regenerated on demand,
  not stored.

### Are we ready for the model factory?

Yes. Every artifact traces to the frozen dataset fingerprint, fitting is train-only
and tested as such, padding/truncation decisions are evidence-based rather than
framework defaults taken on faith, and determinism holds across repeated runs.
Nothing here trains a model or touches M0-M4/D0 specifications.